In [109]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [110]:
import os,sys
#sys.path.append('/work/qdiff/mo_utils')
sys.executable


'/home/nadavg/anaconda3/envs/qdiff/bin/python'

In [111]:
os.environ['CUDA_VISIBLE_DEVICES'] = '3'

In [112]:
from mo_utils.utils.stand_alone_utils.har_utils import get_har_files,get_params_from_har
from mo_utils.utils.stand_alone_utils.pytorch2accelras import get_nested_attr
from mo_utils.utils.stand_alone_utils.quant_utils import calc_snr

In [113]:
from src.utils.torch_utils import add_full_name_to_module

In [114]:
import netron
from pathlib import Path

In [115]:
from diffusers import StableDiffusionPipeline

In [116]:
pipe = StableDiffusionPipeline.from_pretrained("SG161222/Realistic_Vision_V4.0_noVAE")

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

In [117]:
pipe.to('cuda')

RuntimeError: No CUDA GPUs are available

In [21]:
unet = pipe.unet
add_full_name_to_module(unet)

In [54]:
ace_unet_path =  '/genai/users/ellaf/stable_diffusion/unet/shared_unet_demo/unet_sim.har'

In [55]:
get_har_files(ace_unet_path)

['unet_sim.hn',
 'unet_sim.npz',
 'unet_sim.original_model_meta.json',
 'unet_sim.metadata.json']

In [56]:
params = get_params_from_har(ace_unet_path,params_name='unet_sim.npz')

all_names=['unet_sim.hn', 'unet_sim.npz', 'unet_sim.original_model_meta.json', 'unet_sim.metadata.json']
loading  params_name='unet_sim.npz' ...


In [57]:
params.keys()

dict_keys(['unet_sim/conv1/padding_const_value:0', 'unet_sim/conv1/kernel:0', 'unet_sim/conv1/bias:0', 'unet_sim/layer_normalization103/epsilon:0', 'unet_sim/normalization103/kernel:0', 'unet_sim/normalization103/bias:0', 'unet_sim/layer_normalization1/epsilon:0', 'unet_sim/normalization1/kernel:0', 'unet_sim/normalization1/bias:0', 'unet_sim/fc1/kernel:0', 'unet_sim/fc1/bias:0', 'unet_sim/conv12/padding_const_value:0', 'unet_sim/conv12/kernel:0', 'unet_sim/conv12/bias:0', 'unet_sim/mul_and_add1/kernel:0', 'unet_sim/mul_and_add1/bias:0', 'unet_sim/conv23/padding_const_value:0', 'unet_sim/conv23/kernel:0', 'unet_sim/conv23/bias:0', 'unet_sim/conv28/padding_const_value:0', 'unet_sim/conv28/kernel:0', 'unet_sim/conv28/bias:0', 'unet_sim/mul_and_add9/kernel:0', 'unet_sim/mul_and_add9/bias:0', 'unet_sim/conv29/padding_const_value:0', 'unet_sim/conv29/kernel:0', 'unet_sim/conv29/bias:0', 'unet_sim/conv30/padding_const_value:0', 'unet_sim/conv30/kernel:0', 'unet_sim/conv30/bias:0', 'unet_sim/

In [58]:
hn = get_params_from_har(ace_unet_path,params_name='unet_sim.hn')

all_names=['unet_sim.hn', 'unet_sim.npz', 'unet_sim.original_model_meta.json', 'unet_sim.metadata.json']
loading  params_name='unet_sim.hn' ...


In [59]:
#[ki for ki in hn['layers'].keys() if 'input' in ki]
#[hn['layers'][ki] for ki in hn['layers'].keys() if 'input' in ki]

In [60]:
k = list(hn['layers'].keys())

In [22]:
k[:10]

['unet_sim/input_layer1',
 'unet_sim/conv1',
 'unet_sim/input_layer2',
 'unet_sim/fc1',
 'unet_sim/fc2',
 'unet_sim/fc10',
 'unet_sim/fc11',
 'unet_sim/fc12',
 'unet_sim/fc13',
 'unet_sim/fc14']

'normalization'

In [70]:
ind = 13
k[ind], hn['layers'][k[ind]]['original_names'],hn['layers'][k[ind]]['type']

('unet_sim/fc18',
 ['/down_blocks.0/resnets.0/time_emb_proj/Gemm',
  '/down_blocks.0/resnets.0/Unsqueeze_1'],
 'dense')

In [182]:
hn['layers'][k[ind]]['original_names']

['/up_blocks.3/resnets.0/time_emb_proj/Gemm',
 '/up_blocks.3/resnets.0/Unsqueeze_1']

In [103]:
#ind =75
ind =13
obj_pa= get_nested_attr(unet,hn['layers'][k[ind]]['original_names'][0],2)
obj= get_nested_attr(unet,hn['layers'][k[ind]]['original_names'][0])
hn['layers'][k[ind]]['original_names'][0], obj_pa.full_name, obj.full_name,  obj

('/down_blocks.0/resnets.0/time_emb_proj/Gemm',
 'down_blocks.0.resnets.0',
 'down_blocks.0.resnets.0.time_emb_proj',
 Linear(in_features=1280, out_features=320, bias=True))

In [104]:
type(obj_pa)#.emb_layers

diffusers.models.resnet.ResnetBlock2D

In [105]:
type(obj)

torch.nn.modules.linear.Linear

In [106]:
kp = [ki for ki in params.keys() if k[ind] in ki]
kp

['unet_sim/fc18/kernel:0', 'unet_sim/fc18/bias:0']

In [107]:
wa = params[kp[0]] 
ba = params[kp[1]]
wa.shape,ba.shape

((1280, 320), (320,))

In [158]:
obj.weight.data.numpy().shape,obj.bias.data.numpy().shape

((320, 1280), (320,))

In [161]:
b.dtype,obj.bias.data.dtype

(dtype('float32'), torch.float32)

In [164]:
dw = w - obj.weight.data.numpy().transpose(1,0)
dw.shape

(1280, 320)

In [168]:
calc_snr( w , obj.weight.data.numpy().transpose(1,0))

8.188028037559079

In [159]:
b- obj.bias.data.numpy()

array([ 2.71775317e-03, -2.89088488e-03,  1.61996484e-03, -3.59436218e-03,
        4.91935760e-04, -1.74783543e-03,  1.01880729e-03, -1.71388453e-03,
        8.96308571e-04,  1.00519508e-04,  4.79545444e-04,  3.14351171e-03,
        3.61159444e-04,  2.40278617e-03,  5.61349094e-04, -5.34360856e-03,
        1.72252208e-03, -4.86560166e-04, -7.16480613e-03,  2.44987290e-03,
       -3.15394253e-04, -1.54741853e-03, -2.98254192e-03, -3.68095934e-05,
        4.78599221e-04, -5.27270138e-04,  2.79220194e-03,  9.91467386e-04,
        1.20504387e-03, -1.53485686e-04, -2.14839354e-03,  2.28279829e-03,
        3.53718176e-03,  3.23645771e-04,  2.16829963e-03, -2.10886449e-03,
        6.33336604e-04, -6.73811883e-04, -4.59407456e-04, -1.91850029e-03,
        7.94373453e-04, -5.06527722e-04, -4.42927703e-05,  7.49802683e-04,
        1.08234584e-04,  1.30308140e-03,  9.33226198e-04, -8.02762806e-04,
       -8.81670788e-03, -6.99314103e-03, -1.17528252e-03,  9.48730856e-04,
       -6.86367974e-04, -

In [130]:
obj= get_nested_attr(unet,hn['layers'][k[ind]]['original_names'][0],ignore_last=2).norm2
obj.weight.shape,obj.bias.shape

(torch.Size([320]), torch.Size([320]))

In [141]:
obj.weight.detach().numpy().shape,w.shape

((320,), (1, 1, 320, 1))

In [144]:
d = obj.weight.detach().numpy()-w[0,0,:,0]
d.shape

(320,)

In [145]:
d

array([-1.37381464e-01, -3.92369330e-02, -4.58287299e-02, -1.92068964e-01,
       -9.70982611e-02, -2.11478144e-01, -1.56180292e-01, -9.97838080e-02,
       -1.75833613e-01, -1.04422480e-01, -7.48814642e-02, -1.50320917e-01,
       -3.97252142e-02, -5.65709174e-02, -8.02525580e-02, -1.38113886e-01,
       -9.24595892e-02, -3.58189642e-02, -7.82994330e-02, -7.98693299e-03,
       -5.68150580e-02, -9.63658392e-02, -1.05887324e-01, -1.27127558e-01,
       -1.40799433e-01, -8.92857611e-02, -1.03445917e-01, -8.90416205e-02,
       -7.58580267e-02, -8.36705267e-02, -1.09793574e-01, -1.95609003e-01,
       -8.83091986e-02, -1.06863886e-01, -1.11014277e-01, -9.97838080e-02,
       -9.63658392e-02, -5.43736517e-02, -9.46568549e-02, -9.17271674e-02,
       -1.52396113e-01, -5.31529486e-02, -1.24930292e-01, -1.30301386e-01,
       -8.19615424e-02, -3.67955267e-02, -2.23912299e-02, -1.10037714e-01,
       -1.22977167e-01, -7.29283392e-02,  2.37513483e-02, -5.07115424e-02,
       -4.97349799e-02, -

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained("SG161222/Realistic_Vision_V4.0_noVAE")

In [92]:
#!/usr/bin/env python
from typing import Optional, Union

import torch
from diffusers import AutoencoderKL, DDIMScheduler, StableDiffusionPipeline

base_model_path = "SG161222/Realistic_Vision_V4.0_noVAE"
orig_base_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"
vae_model_path = "stabilityai/sd-vae-ft-mse"
device = "cuda"
dtype = torch.float32

scheduler = DDIMScheduler.from_pretrained(base_model_path, subfolder="scheduler")
vae = AutoencoderKL.from_pretrained(vae_model_path, torch_dtype=dtype)
pipe = StableDiffusionPipeline.from_pretrained(
    orig_base_model_path,
    torch_dtype=dtype,
    scheduler=scheduler,
    vae=vae,
    # feature_extractor=AutoFeatureExtractor.from_pretrained(
    #     orig_base_model_path, subfolder="feature_extractor", torch_dtype=dtype
    # ),
    safety_checker=None,
)

unet = pipe.unet
add_full_name_to_module(unet)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [93]:
import onnx

In [94]:
onnx_path = '/genai/users/shacharg/stable_diffusion/2025-02-06/unet_sim.onnx'

In [95]:
onnx_path = '/genai/users/shacharg/stable_diffusion/2025-02-06/unet_sim.onnx'
onnx_model = onnx.load(onnx_path)

In [96]:
type(onnx_model)

onnx.onnx_ml_pb2.ModelProto

In [97]:
#/up_blocks.3/resnets.0/time_emb_proj/#''

In [98]:
onnx_model.graph.node[0].input


['/time_proj/Concat_1_output_0', 'time_embedding.linear_1.weight', 'time_embedding.linear_1.bias']

In [99]:
import numpy as np
for initializer in onnx_model.graph.initializer:
    if "time_emb_proj" in initializer.name:
        weight_array = np.frombuffer(initializer.raw_data, dtype=np.float32).reshape(initializer.dims)
        print(f"Found weight for {initializer.name} with shape {initializer.dims}")
        break

Found weight for down_blocks.0.resnets.0.time_emb_proj.weight with shape [320, 1280]


In [100]:
w= unet.down_blocks[0].resnets[0].time_emb_proj.weight.data.numpy()

In [101]:
weight_array.shape

(320, 1280)

In [102]:
calc_snr(weight_array,w)

118.30428327609846

In [91]:
wa.shape , wa.transpose(1,0).shape,w.shape

((1280, 320), (320, 1280), (320, 1280))

In [108]:
#wa.transpose(1,0).shape
calc_snr(w,wa.transpose(1,0))

118.30428327609846

In [53]:
weight_array-w

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)